<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">

# VISTA-3D NIM

This notebook will walk you through how to start VISTA-3D NIM and run your first inference.

<p align = "center"><img src="https://catalog.ngc.nvidia.com/demos/vista.jpg" width="400"/></p>

### Table of Contents
**[0.0 NIM Setup](#0.0-NIM-Setup)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[0.1 NIM Background](#0.1-NIM-Background) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[0.2 VISTA-3D](#0.2-VISTA-3D) <br>
**[1.0 Download and Start VISTA-3D NIM](#1.0-Download-and-Start-VISTA-3D-NIM)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.1 Download the NIM Container](#1.1-Download-the-NIM-Container) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.2 Set Up Directories for the NIM](#1.2-Set-Up-Directories-for-the-NIM) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.3 Start the NIM Server](#1.3-Start-the-NIM-Server) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.4 Verify NIM Startup](#1.4-Verify-NIM-Startup) <br>
**[2.0 Run Inference with VISTA-3D](#2.0-Run-Inference-with-VISTA-3D)** <br>
**[3.0 Integrating with MONAI Label](#3.0-Integrating-with-MONAI-Label)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[3.1 Config-File](#3.1-Config-File) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[3.2 Write `infer/vista3d_nim`](#3.2-Write-infer/vista3d_nim) <br>
**[4.0 Open and Test with Slicer](#4.0-Open-and-Test-with-Slicer)** <br>
**[5.0 Start MONAI Label Server](#5.0-Start-MONAI-Label-Server)** <br>
**[6.0 Clean Up Docker](#6.0-Clean-Up-Docker)** <br>
**[7.0 VISTA-3D References](#7.0-VISTA-3D-References)** <br>
**[8.0 NIM Exercise](#8.0-NIM-Exercise)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[8.1 Add More Labels for VISTA-3D to Segment](#8.1-Add-More-Labels-for-VISTA-3D-to-Segment) <br>

---
# 0.0 NIM Setup 

lets import some libs used in this notebook.

In [ ]:
import ipywidgets
import os
HOST_NAME=os.getenv('HOST_NAME')
HOST_IP=os.getenv('HOST_IP')
DLI_DATA_DIR=os.getenv('DLI_DATA_DIR')
os.chdir(DLI_DATA_DIR)
RADOLOGY_APP_ROOT=DLI_DATA_DIR+"/apps/radiology"

if len(HOST_NAME)==0:
    from requests import get
    HOST_NAME=get('https://api.ipify.org').text
    HOST_IP=HOST_NAME
print(f" HOST_NAME is {HOST_NAME}  with IP {HOST_IP} ")

## 0.1 NIM Background 

NIM stands for NVIDIA Inference Microservice, announced in 2024 at the NVIDIA GTC Conference. 
The diagram below shows the architecture for the NIMs where AI models are optimized to run on docker or [Helm](https://helm.sh/). 
It follows the OpenAI standard API.
<p align = "center"><img src="./images/Nvidia_nim.png" width="900"/></p>

You can test out any of the NIMs at [build.nvidia.com](https://build.nvidia.com/).

## 0.2 VISTA-3D

VISTA-3D is a specialized interactive foundation model for segmenting and annotating human anatomies. VISTA-3D segments 126 classes including 5 diseases (liver, kidney, pancreatic, lung, colon, and bone tumor)

VISTA-3D has three core workflows:
- Segment everything: Enables whole body exploration, crucial for understanding complex diseases affecting multiple organs and for holistic treatment planning.
- Segment using class: Provides detailed sectional views based on specific classes, essential for targeted disease analysis or organ mapping, such as tumor identification in critical organs.
- Segment point prompts: Enhances segmentation precision through user-directed, click-based selection. This interactive approach accelerates the creation of accurate ground-truth data, essential in medical imaging analysis.

You can test out VISTA-3D NIM at https://build.nvidia.com/nvidia/vista-3d.

<p align = "center"><img src="./images/Vista_nim_build.nvidia.com.png" width="800"/></p>

---
# 1.0 Download and Start VISTA-3D NIM
We will need to download the NIM container and start it.

## 1.1 Download the NIM Container
We have already downloaded the container for you.
If you want to use it on your local infrastructure,
you should be able to download the Docker container by running cell below.

In [ ]:
# ! docker pull nvcr.io/nim/nvidia/vista3d:1.0

If you face an authentication error such as `Error response from daemon: unauthorized: authentication required`, 
this indicates you need to first log into the NGC registry using your NGC API Key.

```bash
docker login nvcr.io --username='$oauthtoken' --password=<paste your ngc API KEY here>
```

You can get the API key:
1. By following steps on the NIM Docker tab [here](https://build.nvidia.com/nvidia/vista-3d?snippet_tab=Docker).
2. Or create an account on NGC, then generate an API key following the steps [here](https://docs.nvidia.com/launchpad/ai/base-command-coe/latest/bc-coe-docker-basics-step-02.html).

## 1.2 Set Up Directories for the NIM
NVIDIA created NIMs so you can run them locally next to your images. 
For that, we need to create directories so the VISTA-3D NIM can read and write from them.

In [ ]:
!mkdir -p /dli/data/vista3d_nim_output
!chmod 777 /dli/data/vista3d_nim_output

NIMs run as docker containers. To do this we need to map the real host path to the data folder. The host pagh value can be found by inspecting the mounts with `docker inspect`.  Run the next cell to see the mounts listed.  We are looking for the source for the `dli/data` destination mount.

In [ ]:
!docker inspect dli-lab | jq '.[].Mounts[] | {Type, Source, Destination}'

To capture the correct value into a Python variable `data_path_on_host` we can use to launch the NIM, execute the following cell.  Verify that that this value is set as we expect.

In [ ]:
import subprocess
import json

# Run docker volume inspect and capture the output
result = subprocess.run(
    ["docker", "volume", "inspect", "s-hx-02-v1_monailabel_data"],
    capture_output=True,
    text=True,
    check=True  # Raises CalledProcessError if the command fails
)

# Parse JSON and extract the mount point
data_path_on_host = json.loads(result.stdout)[0]["Mountpoint"]

print("Volume Mountpoint:", data_path_on_host)


## 1.3 Start the NIM Server
Execute the cell below to start the NIM.
If you run this on your own, add your API key as 
`-e NGC_API_KEY=<your API key> \` 
because the model needs to download the model weights from NGC.

For this lab, we have already done this for you.

In [ ]:
!docker run -d --rm \
   --name vista3d_nim \
    --network=dli-network \
    -e NIM_HTTP_API_PORT=8008 -p 8008:8008 \
    -e NIM_CACHE_PATH="/vista_nim_model" \
    -v $data_path_on_host/vista_nim_model:/vista_nim_model \
    -e TRT_INFERENCE=False \
      -e IMAGE_URI_HTTPS_ONLY=False \
      -e IGNORE_SSL_ERRORS=True \
      -e IMAGE_URI_ALLOW_REDIRECTS=True \
      -e DOMAIN_WHITELIST='["http://*","https://*","/dli/data/*"]' \
   --runtime=nvidia \
   --shm-size=8G \
   -v $data_path_on_host/:/dli/data/ \
   nvcr.io/nim/nvidia/vista3d:1.0

## 1.4 Verify NIM Startup
Lets see if the VISTA-3D NIM container is running.

In [ ]:
!docker ps | grep vista3d_nim

We will now use the microservice API to check on the health of the NIM. 

<div class="alert alert-block alert-info">
This can take up to 4 min as model is loaded into triton. 

    You may get message as:
1. connection was reset 
2. curl: (7) Failed to connect to vista3d_nim port 8008 after 0 ms: Connection refused

<b>Please keep trying until you get {"message":"Both triton server and vista3d model are ready!"}.</b>
</div>

In [ ]:
# !curl $HOST_NAME:8008/v1/health/ready
!curl http://vista3d_nim:8008/v1/health/ready

In [ ]:
## you could use python code to check on health as below
import requests
url = "http://vista3d_nim:8008/v1/health/ready"
headers = {
    "accept": "application/octet-stream",
    "content-type": "application/json"
}
response = requests.get(url, headers=headers)
print(response.text)

---
# 2.0 Run Inference with VISTA-3D

In [ ]:
### trying in python
import requests

file_name="spleen_8.nii.gz"

invoke_url = "http://vista3d_nim:8008/v1/vista3d/inference"
headers = {"Content-Type": "application/json"}
payload = {
    # "image": f"http://{HOST_NAME}:7089/{file_name}",
    "image":f"/dli/data/datasets/Task09_Spleen/imagesTr/{file_name}",
    "local_working_dir": "/dli/data/vista3d_nim_output",
    "prompts": {"classes": ["liver", "spleen"] },
  "output": {"extension": ".nii.gz", "dtype": "uint8"}
}
## make call 
response = requests.post(invoke_url, headers=headers, json=payload)
# Save the content to a file
if response.ok:
    with open(file_name, "wb") as file:
        file.write(response.content)
        print("File downloaded successfully!")
        print(response.text)
        print(response.json()['message'])
else:
    print(f"error in response {response.text}")

---
# 3.0 Integrating with MONAI Label 

Now that we have successfully used VISTA-3D NIM, let's try and connect it to MONAI Label and use it in Slicer. 
Below is how the architecture looks.

<p align = "center"><img src="./images/monailabel_arch_vista3d_nim.png" width="800"/></p>
We will continue creating a simple app to make a call to VISTA-3D NIM. 
We'll be building on the radiology app, just adding a few files to call into VISTA-3D NIM.

```bash
App
├── lib/infers 
├── lib/configs 
```

## 3.1 Config File
Run the cell below to write a file for `config/my_spleen`.

In [ ]:
%%writefile {RADOLOGY_APP_ROOT}/lib/configs/vista3d_nim.py
from typing import Any, Dict, Optional, Union
from monailabel.interfaces.config import TaskConfig
from monailabel.interfaces.tasks.infer_v2 import InferTask
from monailabel.interfaces.tasks.train import TrainTask
import logging, os
logger = logging.getLogger(__name__)

class Vista3d_NIM(TaskConfig):    
    def init(self, name: str, model_dir: str, conf: Dict[str, str], planner: Any, **kwargs):
        super().init(name, model_dir, conf, planner, **kwargs)          
        self.labels = {
            "liver": 1,
            "kidney": 2,
            "spleen": 3,
            "pancreas": 4,
            "right kidney": 5,
            "aorta": 6,
            "inferior vena cava": 7,
            "right adrenal gland": 8,
            "left adrenal gland": 9,
            "gallbladder": 10,
            "esophagus": 11,
            "stomach": 12,
            "duodenum": 13,
            "left kidney": 14,
            "bladder": 15,
            "prostate or uterus": 16,
            "portal vein and splenic vein": 17,
            "rectum": 18,  # gives error 
            "small bowel": 19,  # gives error 
            "lung": 20,
            "bone": 21,
            "brain": 22,
            "lung tumor": 23,
            "pancreatic tumor": 24,
            "hepatic vessel": 25,
            "hepatic tumor": 26,
            "colon cancer primaries": 27,
            "left lung upper lobe": 28,
            "left lung lower lobe": 29,
            "right lung upper lobe": 30,
            "right lung middle lobe": 31,
            "right lung lower lobe": 32,
            "vertebrae L5": 33,
            "vertebrae L4": 34,
            "vertebrae L3": 35,
            "vertebrae L2": 36,
            "vertebrae L1": 37,
            "vertebrae T12": 38,
            "vertebrae T11": 39,
            "vertebrae T10": 40,
            "vertebrae T9": 41,
            "vertebrae T8": 42,
            "vertebrae T7": 43,
            "vertebrae T6": 44,
            "vertebrae T5": 45,
            "vertebrae T4": 46,
            "vertebrae T3": 47,
            "vertebrae T2": 48,
            "vertebrae T1": 49,
            "vertebrae C7": 50,
            "vertebrae C6": 51,
            "vertebrae C5": 52,
            "vertebrae C4": 53,
            "vertebrae C3": 54,
            "vertebrae C2": 55,
            "vertebrae C1": 56,
            "trachea": 57,
            "left iliac artery": 58,
            "right iliac artery": 59,
            "left iliac vena": 60,
            "right iliac vena": 61,
            "colon": 62,
            "left rib 1": 63,
            "left rib 2": 64,
            "left rib 3": 65,
            "left rib 4": 66,
            "left rib 5": 67,
            "left rib 6": 68,
            "left rib 7": 69,
            "left rib 8": 70,
            "left rib 9": 71,
            "left rib 10": 72,
            "left rib 11": 73,
            "left rib 12": 74,
            "right rib 1": 75,
            "right rib 2": 76,
            "right rib 3": 77,
            "right rib 4": 78,
            "right rib 5": 79,
            "right rib 6": 80,
            "right rib 7": 81,
            "right rib 8": 82,
            "right rib 9": 83,
            "right rib 10": 84,
            "right rib 11": 85,
            "right rib 12": 86,
            "left humerus": 87,
            "right humerus": 88,
            "left scapula": 89,
            "right scapula": 90,
            "left clavicula": 91,
            "right clavicula": 92,
            "left femur": 93,
            "right femur": 94,
            "left hip": 95,
            "right hip": 96,
            "sacrum": 97,
            "left gluteus maximus": 98,
            "right gluteus maximus": 90,
            "left gluteus medius": 100,
            "right gluteus medius": 101,
            "left gluteus minimus": 102,
            "right gluteus minimus": 103,
            "left autochthon": 104,
            "right autochthon": 105,
            "left iliopsoas": 106,
            "right iliopsoas": 107,
            "left atrial appendage": 108,
            "brachiocephalic trunk": 109,
            "left brachiocephalic vein": 110,
            "right brachiocephalic vein": 111,
            "left common carotid artery": 112,
            "right common carotid artery": 113,
            "costal cartilages": 114,
            "heart": 115,
            "left kidney cyst": 116,
            "right kidney cyst": 117,
            "prostate": 118,
            "pulmonary vein": 119,
            "skull": 120,
            "spinal cord": 121,
            "sternum": 122,
            "left subclavian artery": 123,
            "right subclavian artery": 124,
            "superior vena cava": 125,
            "thyroid gland": 126,
            "vertebrae S1": 127,
            "bone lesion": 128,
            "kidney mass": 129,
            "liver tumor": 130,
            "vertebrae L6": 131,
            "airway": 132
        }                    
        self.path = [] # Model Files
        self.network = "no model"# Network

    def infer(self) -> Union[InferTask, Dict[str, InferTask]]:
        logger.info(f"-------------------------------- in infer of the config file ")
        from lib.infers.vista3d_nim import Vista3d_NIM
        task: InferTask = Vista3d_NIM(
            path=[],
            network=self.network,
            labels=self.labels,
        )
        return task

    def trainer(self) -> Optional[TrainTask]:
        pass

## 3.2 Write `infer/vista3d_nim`

In [ ]:
%%writefile {RADOLOGY_APP_ROOT}/lib/infers/vista3d_nim.py

from typing import Any, Callable, Sequence, Tuple, Union, Dict, List
from monailabel.interfaces.tasks.infer_v2 import InferType
from monailabel.tasks.infer.basic_infer import BasicInferTask
import os,logging,requests, json, glob
logger = logging.getLogger(__name__)

class Vista3d_NIM(BasicInferTask):
    def __init__(self,path,network=None,type=InferType.SEGMENTATION,
        labels=None,dimension=3,description="Shell class to call VISTA-3D NIM",
        **kwargs,
    ):
        super().__init__(path=path,network="no network",
            type=type,labels=labels,dimension=dimension,
            description=description,**kwargs,
        )
        self.nim_url = "http://vista3d_nim:8008/v1/vista3d/inference"            
        self.local_working_dir= "/dli/data/vista3d_nim_output"
        
    def pre_transforms(self, data=None) -> Sequence[Callable]:
        return []
    def post_transforms(self, data=None) -> Sequence[Callable]:
        return []

    def __call__(self, request, callbacks: Union[Dict, None] = None) -> Union[Dict, Tuple[str, Dict[str, Any]]]:
        logger.info(f"___________________________in my call, request {request} _ open dicom to check on metadata")
        imagePath= request["image"]
        result_file_name = self.curl_post_and_download(imagePath)
        logger.info(f"File saved as: {os.path.abspath(result_file_name)}")
        result_json = {}
        return result_file_name, result_json

    def curl_post_and_download(self, imagePath, classes_lst=None):        
        if not classes_lst:
            classes_lst = [ "spleen","liver" ,"hepatic tumor","right kidney","left kidney" 
                           # ,"stomach", "superior vena cava", "aorta","portal vein and splenic vein" ##Exercise 8.1
                           # ,"left kidney cyst","right kidney cyst"     ##Exercise 8.1
                          ]  
            # classes_lst = []  ## pass empty list to ask for all labels ##Exercise 8.2
        data = { "image": imagePath
                ,"local_working_dir": self.local_working_dir
                , "prompts": { "classes": classes_lst }
                 ,"output": {"extension": ".nii.gz", "dtype": "uint8"}}
        headers = {"Content-Type": "application/json"}
        try:            
            if isinstance(data, dict): # Prepare the data for POST request
                data = json.dumps(data)
            logger.info(f"____________calling vista3d nim _______ {self.nim_url}, {data} ")
            response = requests.post(self.nim_url, data=data, headers=headers, stream=True)
            response.raise_for_status() # Raise an exception for bad status codes
            response = response.json()
            logger.info(f" ------------>>>>>>>>>>>>>>>>>>returned from nim {response}")
            directory_path = response["message"].split("results are saved in ")[-1]            
            # Find the .nii.gz file in the directory
            nii_gz_files = glob.glob(os.path.join(directory_path, "*.nii.gz"))            
            nii_gz_file=""
            if len(nii_gz_files) == 1:
                nii_gz_file = nii_gz_files[0]
            else:
                logger.error("No .nii.gz file found in the directory. Error: More than one .nii.gz file found:", nii_gz_files)
            logger.info(f" ---->>>>>>>>>>>>>>>>>>returned file {nii_gz_file}")
            return nii_gz_file
        except requests.exceptions.RequestException as e:
            logger.error(f"An error occurred: {e}")
            return None

---
# 4.0 Open and Test with Slicer
Please use the Slicer previously launched in the [earlier notebook](01_3dslicer.ipynb).
<div class="alert alert-block alert-info">
Password:<b>gtc25</b>
</div>

In [ ]:
url_text='<h1><center><a href="http://'+HOST_NAME+'/x11/" target="_blank">'
url_text=url_text+'<img src="https://user-images.githubusercontent.com/15837524/103544638-6999fa80-4e6e-11eb-94c4-5c42cd2f63f8.png" width="200" height="100"/>'
url_text=url_text+'<br>Click to open 3D Slicer <br> in a new window</a></center></h1>'
ipywidgets.HTML(url_text)

---
# 5.0 Start MONAI Label Server
Now we need to start the MONAI Label server with our model config using the radiology app.

In [ ]:
%cd $DLI_DATA_DIR
!monailabel start_server --app apps/radiology \
            --studies datasets/Task09_Spleen/imagesTr \
            --conf models vista3d_nim #,segmentation_spleen

---
# 6.0 Clean Up Docker 
Stop the VISTA-3D NIM container by executing the cell below.

In [ ]:
!docker stop vista3d_nim

---
# 7.0 VISTA-3D References

- [VISTA-3D playground](https://build.nvidia.com/nvidia/vista-3d)
- [VISTA-3D performance](https://docs.nvidia.com/nim/medical/vista3d/latest/performance.html)

---
# 8.0 NIM Exercise

go back to section [3.2 Write `infer/vista3d_nim`](#3.2-Write-infer/vista3d_nim) and modify the labels you ask VISTA-3D NIM to segment.

Make sure you have MONAI LABEL server stopped then you run your cell to overwrite the file.

## 8.1 Add More Labels for VISTA-3D to Segment 
You can uncomment some of the labels in the section below. 

Hint: Look at the full list in the config file to add more labels
```
            classes_lst = [ "spleen","liver" ,"hepatic tumor","right kidney","left kidney" 
                           # ,"stomach", "superior vena cava", "aorta","portal vein and splenic vein"
                           # ,"kidney mass","left kidney cyst","right kidney cyst" 
                          ]
```

You can also just pass and empty list in the prompts to ask for all labels. 
```
            classes_lst = []  ## pass empty list to ask for all labels 
```

<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">